# Analisi di Interpretabilità SHAP

🇮🇹 [🇬🇧](06_interpretability.ipynb)

## Predizione del Rischio di Infortunio nei Runner — Replica di Lovdal et al. (2021)

Questo notebook applica **SHAP (SHapley Additive exPlanations)** ai modelli XGBoost
ottimizzati dei notebook 04 e 05. I valori SHAP decompongono ogni predizione in
contributi per-feature, fornendo interpretabilità sia **globale** (quali feature contano
di più su tutte le predizioni) che **locale** (perché una predizione specifica è stata fatta).

### Contenuti

1. Caricamento dati — modelli + test set per approcci giornaliero e settimanale
2. Calcolo dei valori SHAP — TreeExplainer per XGBoost
3. Importanza globale delle feature — beeswarm summary plot
4. Dependence plot — top 5 feature per approccio
5. Predizioni individuali — waterfall plot (casi TP, FN, TN)
6. Confronto importanza feature — SHAP vs XGBoost gain
7. Confronto incrociato giornaliero vs settimanale
8. Interpretazione di dominio — pattern di allenamento e rischio di infortunio

### Concetti chiave

- I **valori SHAP** si basano sulla teoria dei giochi cooperativi (valori di Shapley):
  il contributo di ogni feature è l'effetto marginale medio su tutte le possibili
  coalizioni di feature.
- **TreeExplainer** calcola valori SHAP esatti per modelli ad albero in tempo
  polinomiale — nessuna approssimazione per campionamento necessaria.
- Per la classificazione binaria, ci concentriamo sui valori SHAP della **classe
  positiva** (injury=1) in tutto il notebook.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.interpretability.shap_analysis import (
    compare_feature_importance,
    compute_shap_values,
    get_shap_importance_dict,
    get_top_features,
    plot_shap_dependence,
    plot_shap_summary,
    plot_shap_waterfall,
)
from src.preprocessing.io import load_model, load_splits
from src.utils.logging_config import setup_logging
from src.utils.plotting import save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

---

## 1. Caricamento Dati

Carichiamo i modelli XGBoost ottimizzati salvati nei notebook 04 e 05, insieme ai
test set preprocessati. L'analisi SHAP viene eseguita sul **test set** per spiegare
come il modello generalizza su atleti mai visti.

In [ ]:
# --- Day approach ---
day_model = load_model("day_best_model")
_, day_test = load_splits(prefix="day")
feature_cols_day = get_feature_columns(day_test)
X_test_day = day_test[feature_cols_day]
y_test_day = day_test[INJURY_COL]

# --- Week approach ---
week_model = load_model("week_best_model")
_, week_test = load_splits(prefix="week")
feature_cols_week = get_feature_columns(week_test)
X_test_week = week_test[feature_cols_week]
y_test_week = week_test[INJURY_COL]

print("Day approach:")
print(f"  Model type: {type(day_model).__name__}")
print(f"  Test set: {X_test_day.shape[0]:,} rows x {X_test_day.shape[1]} features")
print(f"  Injury rate: {y_test_day.mean():.2%}")

print("\nWeek approach:")
print(f"  Model type: {type(week_model).__name__}")
print(f"  Test set: {X_test_week.shape[0]:,} rows x {X_test_week.shape[1]} features")
print(f"  Injury rate: {y_test_week.mean():.2%}")

---

## 2. Calcolo dei Valori SHAP

TreeExplainer calcola valori SHAP esatti per XGBoost in tempo polinomiale.
Per la classificazione binaria, l'output ha forma `(n_samples, n_features, 2)` —
estraiamo la classe positiva (injury=1) per tutte le visualizzazioni.

In [ ]:
# Compute SHAP values for both approaches
shap_values_day = compute_shap_values(day_model, X_test_day)
shap_values_week = compute_shap_values(week_model, X_test_week)

print(f"Day SHAP values shape: {shap_values_day.shape}")
print(f"Week SHAP values shape: {shap_values_week.shape}")

---

## 3. Importanza Globale delle Feature — Approccio Giornaliero

Il beeswarm (summary) plot mostra **tutte le feature ordinate per |valore SHAP| medio**.
Ogni punto rappresenta un campione — la posizione orizzontale indica il contributo SHAP
e il colore indica il valore della feature (rosso = alto, blu = basso). Questo rivela
sia **quali feature contano** sia **come i loro valori influenzano le predizioni**.

In [ ]:
# Beeswarm summary — Day approach
fig_summary_day = plot_shap_summary(
    shap_values_day, X_test_day,
    save_path=Path("interpretability/06_shap_summary_day"),
)
plt.show()
plt.close(fig_summary_day)

### Approccio giornaliero — feature principali

Il beeswarm plot sopra rivela quali feature di allenamento giornaliero il modello utilizza
maggiormente per predire il rischio di infortunio. Pattern chiave da osservare:
- **Recenza temporale**: le feature del giorno 0 (oggi) dominano, oppure i giorni
  precedenti (giorno 5-6, una settimana fa) portano più segnale?
- **Tipo di feature**: quali metriche di allenamento (km, sforzo percepito, recupero, ecc.)
  sono più predittive?
- **Direzione**: i punti rossi (valori alti) che spingono verso destra indicano che **più**
  di quella feature aumenta il rischio di infortunio.

---

## 4. Importanza Globale delle Feature — Approccio Settimanale

In [ ]:
# Beeswarm summary — Week approach
fig_summary_week = plot_shap_summary(
    shap_values_week, X_test_week,
    save_path=Path("interpretability/06_shap_summary_week"),
)
plt.show()
plt.close(fig_summary_week)

### Approccio settimanale — feature principali

L'approccio settimanale aggrega i dati di allenamento su finestre di 3 settimane
(settimana 0 = corrente, settimana 1 = precedente, settimana 2 = due settimane fa).
Differenze chiave rispetto all'approccio giornaliero:
- Le feature includono **22 metriche aggregate per settimana** più **3 rapporti km relativi**
- I rapporti relativi (carico acuto:cronico) sono un indicatore di rischio di infortunio
  ben noto nella scienza dello sport
- Il segnale temporale è più grossolano ma può catturare pattern di accumulo del carico
  a lungo termine

---

## 5. SHAP Dependence Plot — Top 5 Feature

I dependence plot mostrano **come il valore di una singola feature influenza il contributo
SHAP** su tutti i campioni del test set. Il colore indica la feature di interazione più
correlata (selezionata automaticamente da SHAP).

In [ ]:
# Dependence plots — Day approach (top 5 features)
top5_day = get_top_features(shap_values_day, n=5)
print(f"Day approach — top 5 features: {top5_day}\n")

for feat in top5_day:
    fig = plot_shap_dependence(
        shap_values_day, X_test_day, feat,
        save_path=Path(f"interpretability/06_dep_day_{feat}"),
    )
    plt.show()
    plt.close(fig)

### Dependence plot giornaliero — interpretazione

I dependence plot sopra mostrano la relazione tra il valore grezzo di ogni feature
principale (asse x) e il suo contributo SHAP alla predizione di infortunio (asse y).

Pattern chiave da osservare:
- **Soglie non lineari**: salti improvvisi nel valore SHAP a valori specifici della feature
  possono indicare soglie di carico di allenamento oltre le quali il rischio di infortunio
  aumenta
- **Effetti di interazione**: i gradienti di colore rivelano quali feature secondarie
  modificano l'effetto della feature principale
- **Cluster a valore zero**: i campioni a 0.0 rappresentano giorni di riposo (sentinella
  −0.01 sostituita con 0.0 nel preprocessing) — formano un cluster distinto

In [ ]:
# Dependence plots — Week approach (top 5 features)
top5_week = get_top_features(shap_values_week, n=5)
print(f"Week approach — top 5 features: {top5_week}\n")

for feat in top5_week:
    fig = plot_shap_dependence(
        shap_values_week, X_test_week, feat,
        save_path=Path(f"interpretability/06_dep_week_{feat}"),
    )
    plt.show()
    plt.close(fig)

---

## 6. Predizioni Individuali — Waterfall Plot

I waterfall plot spiegano **predizioni individuali** mostrando come ogni feature
sposta l'output del modello dal valore base (media della popolazione) al punteggio
finale del modello per quel caso. Con `shap.TreeExplainer(model)` e impostazioni
predefinite, questo è tipicamente l'output grezzo del modello (ad esempio, log-odds
nella classificazione binaria) piuttosto che una probabilità. Selezioniamo tre
casi rappresentativi per approccio:

- **True Positive (TP)**: infortunio predetto correttamente — quali feature hanno guidato l'allarme?
- **False Negative (FN)**: infortunio mancato — perché il modello ha fallito?
- **True Negative (TN)**: non-infortunio predetto correttamente — confronto di baseline

In [ ]:
def select_case_indices(
    y_true: pd.Series, y_prob: np.ndarray
) -> dict[str, int]:
    """Select representative TP, FN, TN indices for waterfall plots.

    For each case type, picks the sample with the most extreme predicted
    probability to make the waterfall plot maximally informative.
    """
    y_pred = (y_prob >= 0.5).astype(int)
    y_arr = np.asarray(y_true)

    # True Positive: actual=1, predicted=1 — highest probability
    tp_mask = (y_arr == 1) & (y_pred == 1)
    # False Negative: actual=1, predicted=0 — lowest probability among positives
    fn_mask = (y_arr == 1) & (y_pred == 0)
    # True Negative: actual=0, predicted=0 — lowest probability
    tn_mask = (y_arr == 0) & (y_pred == 0)

    cases = {}
    if tp_mask.any():
        cases["TP"] = int(np.nonzero(tp_mask)[0][np.argmax(y_prob[tp_mask])])
    if fn_mask.any():
        cases["FN"] = int(np.nonzero(fn_mask)[0][np.argmin(y_prob[fn_mask])])
    if tn_mask.any():
        cases["TN"] = int(np.nonzero(tn_mask)[0][np.argmin(y_prob[tn_mask])])

    return cases


# --- Day approach: select cases ---
y_prob_day = day_model.predict_proba(X_test_day)[:, 1]
day_cases = select_case_indices(y_test_day, y_prob_day)

print("Day approach — selected cases for waterfall plots:")
for case_type, idx in day_cases.items():
    print(f"  {case_type}: index={idx}, y_true={y_test_day.iloc[idx]}, "
          f"y_prob={y_prob_day[idx]:.4f}")

In [ ]:
# Waterfall plots — Day approach
for case_type, idx in day_cases.items():
    print(f"\n--- {case_type} case (index={idx}) ---")
    fig = plot_shap_waterfall(
        shap_values_day, index=idx,
        save_path=Path(f"interpretability/06_waterfall_day_{case_type.lower()}"),
    )
    plt.show()
    plt.close(fig)

### Waterfall plot giornaliero — interpretazione

I waterfall plot sopra decompongono tre predizioni individuali:

- **TP (True Positive)**: il modello ha identificato correttamente un rischio di infortunio.
  Le feature che spingono la predizione sopra il valore base rivelano quali pattern di
  allenamento il modello associa a un infortunio imminente.
- **FN (False Negative)**: il modello ha mancato questo infortunio. Capire quali feature
  hanno mantenuto bassa la predizione aiuta a identificare i punti ciechi del modello —
  forse l'infortunio aveva un pattern insolito (es. non preceduto da un carico elevato).
- **TN (True Negative)**: una baseline di salute. Le feature dovrebbero generalmente
  spingere la predizione sotto il valore base, confermando la logica del modello.

In [ ]:
# --- Week approach: select cases ---
y_prob_week = week_model.predict_proba(X_test_week)[:, 1]
week_cases = select_case_indices(y_test_week, y_prob_week)

print("Week approach — selected cases for waterfall plots:")
for case_type, idx in week_cases.items():
    print(f"  {case_type}: index={idx}, y_true={y_test_week.iloc[idx]}, "
          f"y_prob={y_prob_week[idx]:.4f}")

In [ ]:
# Waterfall plots — Week approach
for case_type, idx in week_cases.items():
    print(f"\n--- {case_type} case (index={idx}) ---")
    fig = plot_shap_waterfall(
        shap_values_week, index=idx,
        save_path=Path(f"interpretability/06_waterfall_week_{case_type.lower()}"),
    )
    plt.show()
    plt.close(fig)

---

## 7. Confronto Importanza Feature — SHAP vs XGBoost Gain

XGBoost fornisce un'importanza delle feature integrata basata sul **gain** (il miglioramento
medio nella funzione di loss quando una feature è usata per lo split). L'importanza SHAP
(|valore SHAP| medio) è fondata teoricamente e considera le interazioni tra feature.

Il confronto rivela:
- **Accordo**: entrambi i metodi classificano le stesse feature come importanti → importanza robusta
- **Disaccordo**: una feature ha un ranking alto nel gain ma basso nello SHAP → potrebbe
  effettuare split frequenti ma con un effetto marginale piccolo, o interagire fortemente
  con altre feature

In [ ]:
# --- Day approach: SHAP vs XGBoost gain ---
shap_imp_day = get_shap_importance_dict(shap_values_day)
builtin_imp_day = dict(zip(feature_cols_day, day_model.feature_importances_))

fig_cmp_day = compare_feature_importance(
    shap_imp_day, builtin_imp_day, top_n=15,
    save_path=Path("interpretability/06_importance_comparison_day"),
)
plt.show()
plt.close(fig_cmp_day)

In [ ]:
# --- Week approach: SHAP vs XGBoost gain ---
shap_imp_week = get_shap_importance_dict(shap_values_week)
builtin_imp_week = dict(zip(feature_cols_week, week_model.feature_importances_))

fig_cmp_week = compare_feature_importance(
    shap_imp_week, builtin_imp_week, top_n=15,
    save_path=Path("interpretability/06_importance_comparison_week"),
)
plt.show()
plt.close(fig_cmp_week)

---

## 8. Confronto Incrociato Giornaliero vs Settimanale

Per confrontare l'importanza delle feature tra approcci (che usano insiemi di feature
diversi), estraiamo il **tipo di feature base** rimuovendo il prefisso temporale
(`day_N_` o `week_N_`). Questo rivela quali metriche di allenamento sono consistentemente
importanti indipendentemente dalla granularità temporale.

In [ ]:
import re


def extract_base_feature(feature_name: str) -> str:
    """Strip temporal prefix (day_N_ or week_N_) to get the base feature type."""
    return re.sub(r"^(day|week)_\d+_", "", feature_name)


def aggregate_importance_by_type(
    importance: dict[str, float],
) -> dict[str, float]:
    """Sum SHAP importance across temporal windows for each base feature type."""
    agg: dict[str, float] = {}
    for feat, val in importance.items():
        base = extract_base_feature(feat)
        agg[base] = agg.get(base, 0.0) + val
    return agg


# Aggregate by base feature type
day_by_type = aggregate_importance_by_type(shap_imp_day)
week_by_type = aggregate_importance_by_type(shap_imp_week)

# Find common feature types
common_types = sorted(set(day_by_type) & set(week_by_type))

print(f"Day unique feature types: {len(day_by_type)}")
print(f"Week unique feature types: {len(week_by_type)}")
print(f"Common feature types: {len(common_types)}")
print(f"\nCommon types: {common_types}")

In [ ]:
# Cross-comparison bar chart: day vs week by base feature type
if common_types:
    # Normalize to percentages for fair comparison (different scales)
    day_total = sum(day_by_type[t] for t in common_types)
    week_total = sum(week_by_type[t] for t in common_types)

    day_pct = {t: day_by_type[t] / day_total * 100 for t in common_types}
    week_pct = {t: week_by_type[t] / week_total * 100 for t in common_types}

    # Sort by average importance
    sorted_types = sorted(
        common_types,
        key=lambda t: (day_pct[t] + week_pct[t]) / 2,
        reverse=True,
    )

    x = np.arange(len(sorted_types))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(x + width / 2, [day_pct[t] for t in sorted_types],
            width, label="Day approach", color="#2196F3", alpha=0.8)
    ax.barh(x - width / 2, [week_pct[t] for t in sorted_types],
            width, label="Week approach", color="#FF9800", alpha=0.8)

    ax.set_yticks(x)
    ax.set_yticklabels(sorted_types)
    ax.set_xlabel("Relative SHAP importance (%)")
    ax.set_title("Feature Type Importance: Day vs Week Approach", fontweight="bold")
    ax.legend()
    ax.invert_yaxis()
    fig.tight_layout()

    save_figure(fig, "06_cross_comparison_day_week", subdir="interpretability",
                close=False)
    plt.show()
    plt.close(fig)

---

## 9. Interpretazione di Dominio: Pattern di Allenamento e Rischio di Infortunio

### Collegare SHAP alla scienza dello sport

L'analisi SHAP colma il divario tra **predizioni ML black-box** e **insight azionabili
della scienza dello sport**. Temi interpretativi chiave:

#### Carico di allenamento e rischio di infortunio

- Le feature di **distanza totale (km)** compaiono probabilmente tra i predittori
  principali. Nella scienza dello sport, picchi acuti nel volume di allenamento sono un
  fattore di rischio di infortunio ben documentato (Gabbett, 2016). SHAP può rivelare
  se il modello ha appreso questa relazione — valori di km elevati nei giorni/settimane
  recenti che spingono le predizioni verso l'infortunio.

- L'**allenamento ad alta intensità** (km_z3_4, km_z5_t1_t2, km_sprinting) cattura
  la qualità del carico di allenamento, non solo il volume. Queste metriche possono
  interagire con total_km — lo stesso volume a intensità più alta potrebbe comportare
  più rischio.

#### Recupero e indicatori soggettivi

- **Recupero percepito** e **sforzo percepito** sono report soggettivi dell'atleta.
  Se si posizionano in alto nell'importanza SHAP, questo valida l'uso dell'auto-
  valutazione dell'atleta come strumento di monitoraggio. Un recupero percepito basso
  combinato con un carico di allenamento elevato è un classico pattern pre-infortunio.

- Il **successo percepito dell'allenamento** potrebbe catturare la fiducia dell'atleta —
  valori più bassi potrebbero indicare segni precoci di sovrallenamento (scarsa performance
  in allenamento).

#### Pattern temporali

- **Approccio giornaliero**: se le feature dei giorni 0-1 (più recenti) dominano sui
  giorni 5-6, il modello si concentra sul carico acuto. Se anche i giorni precedenti
  contano, il modello cattura la fatica cumulativa.

- **Approccio settimanale**: i rapporti km relativi quantificano direttamente il rapporto
  carico acuto-cronico. Valori > 1.0 indicano un picco di allenamento rispetto alla
  baseline — un noto trigger di infortunio nella letteratura.

#### Confronto con Lovdal et al. (2021)

Lo studio originale riportava che le feature di volume di allenamento e gli indicatori
soggettivi erano i predittori più importanti. La nostra analisi SHAP può confermare
(o contestare) questi risultati con un metodo di attribuzione delle feature più rigoroso.

### Limitazioni

- SHAP spiega il **modello**, non il meccanismo causale sottostante. Un'alta importanza
  SHAP per una feature significa che il modello la usa, non che causa l'infortunio.
- La sostituzione del valore sentinella (ADR-007: −0.01 → 0.0) implica che i giorni di
  riposo hanno valori zero per tutte le feature di allenamento — il modello potrebbe
  imparare a usare i "cluster di zeri" come segnale, che è un pattern valido ma indiretto.
- TreeExplainer fornisce valori SHAP esatti per il modello XGBoost ma non tiene conto
  di input fuori distribuzione o dell'incertezza del modello.

---

## 10. Riepilogo

### Flusso della pipeline

```
data/processed/day_best_model.pkl + week_best_model.pkl
  → load_model()
data/processed/day_test.parquet + week_test.parquet
  → load_splits()
  → compute_shap_values() [TreeExplainer]
  → Globale: beeswarm summary plot
  → Per-feature: dependence plot (top 5)
  → Individuale: waterfall plot (TP, FN, TN)
  → Confronto: SHAP vs XGBoost gain
  → Confronto incrociato: giornaliero vs settimanale per tipo di feature
  → Tutte le figure salvate in reports/figures/interpretability/
```

### Risultati principali

1. **Importanza globale**: i beeswarm plot identificano quali feature di allenamento
   i modelli XGBoost utilizzano maggiormente per la predizione degli infortuni
2. **Dependence plot**: rivelano relazioni non lineari ed effetti di interazione
   tra le feature di carico di allenamento e il rischio di infortunio
3. **Waterfall plot**: decompongono predizioni individuali, mostrando perché il modello
   ha segnalato (o mancato) specifici casi di infortunio
4. **SHAP vs gain**: valida che i ranking di importanza basati su SHAP e quelli integrati
   sono sostanzialmente coerenti (o evidenzia differenze significative)
5. **Giornaliero vs settimanale**: identifica quali metriche di allenamento sono
   consistentemente importanti attraverso le granularità temporali

### Figure generate

Tutte le figure salvate in `reports/figures/interpretability/`:
- 2 beeswarm summary plot (giornaliero + settimanale)
- 10 dependence plot (5 per approccio)
- 6 waterfall plot (3 per approccio: TP, FN, TN)
- 2 grafici di confronto SHAP-vs-gain
- 1 grafico di confronto incrociato (giornaliero vs settimanale)

### Prossimi passi

- **Notebook 07**: Analisi di fairness — metriche per-gruppo usando gruppi proxy
  (volume, storia infortuni, densità dati)
- **Notebook 08**: Riepilogo comparativo — giornaliero vs settimanale affiancati,
  confronto con benchmark dello studio, raccomandazione finale